# CMM-Scan Segmentierung – Validierung

Validiert die Segmentation-Pipeline (AP2.1 Phase 3) auf realen CMM-Scandaten.

**Labels:**
- `0` background (Werkstück-Oberseite, Umgebung)
- `1` flank_a (linke V-Fase, n_x > 0)
- `2` flank_b (rechte V-Fase, n_x < 0)
- `3` gap_region (freier Raum zwischen Fasen)
- `4` sub_gap_artifacts (Durchstich-Punkte unterhalb der Fasen)

**Ablauf:** Scan laden → Preprocessing → Segmentation per YAML → Step-für-Step-Visualisierung → Metriken → Querschnittsplots → Parameter-Sandkasten.


## 1 – Setup & Imports

In [ ]:
import sys
from pathlib import Path

import numpy as np
import open3d as o3d
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.lines import Line2D

project_root = Path().resolve().parent
if str(project_root / 'src') not in sys.path:
    sys.path.insert(0, str(project_root / 'src'))

from schweiss_ki.preprocessing import PreprocessingPipeline
from schweiss_ki.segmentation import (
    LABELS, NAME_TO_ID, UNLABELED,
    BackgroundRemover, FlankSegmenter, GapClassifier,
    SegmentationPipeline,
)

plt.rcParams['figure.dpi'] = 200
plt.rcParams['savefig.dpi'] = 200

# Farbkonvention für Labels
LABEL_COLORS = {
    -1: '#000000',           # unlabeled (sollte nicht auftreten nach process())
    0:  '#B0BEC5',           # background
    1:  '#1976D2',           # flank_a
    2:  '#F57C00',           # flank_b
    3:  '#7B1FA2',           # gap_region
    4:  '#E53935',           # sub_gap_artifacts
}

def color_array(labels: np.ndarray) -> np.ndarray:
    """Labels → RGB-Array shape (N, 3) für matplotlib scatter."""
    from matplotlib.colors import to_rgb
    return np.array([to_rgb(LABEL_COLORS[int(l)]) for l in labels])

def legend_handles(labels_present: set) -> list:
    """Matplotlib-Legend-Handles für die vorhandenen Labels."""
    handles = []
    for lid in sorted(labels_present):
        name = LABELS.get(lid, 'unlabeled' if lid == UNLABELED else f'?{lid}')
        handles.append(Line2D([0], [0], marker='o', color='w',
                              markerfacecolor=LABEL_COLORS[int(lid)],
                              markersize=8, label=f'[{lid}] {name}'))
    return handles

print('Imports OK')


## 2 – Scan laden & Preprocessing

Lädt den Roh-Scan und führt die komplette Preprocessing-Pipeline aus (wie in `pipeline.yaml` konfiguriert, `source_type='real'`).


In [ ]:
SCAN_FILE = Path('../data/raw/cmm_scans/SCHWEIßSPALT_1,0_auf_2,5.xyz')
CONFIG    = Path('../configs/pipeline.yaml')

assert SCAN_FILE.exists(), f'Scan nicht gefunden: {SCAN_FILE}'
assert CONFIG.exists(),    f'Config nicht gefunden: {CONFIG}'

pcd_raw = o3d.io.read_point_cloud(str(SCAN_FILE))
print(f'Roh geladen: {len(pcd_raw.points):,} Punkte')


In [ ]:
preproc = PreprocessingPipeline.from_config(CONFIG, source_type='real')
print(f'Preprocessing: {preproc}')

pcd, prep_report = preproc.process(pcd_raw)
print(f'Nach Preprocessing: {len(pcd.points):,} Punkte '
      f'({prep_report.total_duration_ms:.0f}ms)')
print(f'Hat Normalen: {pcd.has_normals()}')


In [ ]:
# --- Fix: Normalen nach außen/oben orientieren ---
# orient_normals_consistent_tangent_plane hat hier nach innen orientiert.
# Majority-Vote: wenn mehr Normalen n_z < 0 haben, alle flippen.
normals = np.asarray(pcd.normals)
if (normals[:, 2] < 0).mean() > 0.5:
    print(f'Vor Flip:  {(normals[:, 2] > 0).mean()*100:.1f}% Normalen mit n_z > 0')
    pcd.normals = o3d.utility.Vector3dVector(-normals)
    normals = np.asarray(pcd.normals)
    print(f'Nach Flip: {(normals[:, 2] > 0).mean()*100:.1f}% Normalen mit n_z > 0')
else:
    print('Normalen-Orientierung ok, kein Flip nötig.')

In [ ]:
# --- Diagnose: wo liegen die Normalen tatsächlich? ---
normals = np.asarray(pcd.normals)
pts_all = np.asarray(pcd.points)

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for i, (comp, label) in enumerate(zip(normals.T, ['n_x', 'n_y', 'n_z'])):
    axes[i].hist(comp, bins=80, color='steelblue', alpha=0.8)
    axes[i].set_xlabel(label)
    axes[i].set_xlim(-1.05, 1.05)
    axes[i].axvline(0, color='gray', linestyle=':', lw=0.8)
    axes[i].grid(alpha=0.3)
axes[0].set_ylabel('Anzahl')
fig.suptitle('Normalenverteilung (Peaks = dominante Flächenorientierungen)')
plt.tight_layout(); plt.show()

# Erwartete Peaks bei V-Naht mit 30°-Flanken (Normalen nach außen):
#   Oberseite:  n_x ≈ 0,     n_z ≈ +1
#   Flanke A:   n_x ≈ +0.866, n_z ≈ +0.5
#   Flanke B:   n_x ≈ -0.866, n_z ≈ +0.5
# Bei 60°-Flanken: n_x ≈ ±0.5, n_z ≈ +0.866

print(f'Anteil |n_z| > 0.9 (horizontal):  {(np.abs(normals[:, 2]) > 0.9).mean()*100:.1f}%')
print(f'Anteil  n_z < -0.5 (nach unten):  {(normals[:, 2] < -0.5).mean()*100:.1f}%')
print(f'Anteil  n_x > +0.3:               {(normals[:, 0] >  0.3).mean()*100:.1f}%')
print(f'Anteil  n_x < -0.3:               {(normals[:, 0] < -0.3).mean()*100:.1f}%')

# Nur Punkte im Nahtbereich ansehen (Z < -0.5 sind V-Naht-Punkte)
naht_mask = pts_all[:, 2] < -0.5
if naht_mask.any():
    n_naht = normals[naht_mask]
    print(f'\nNur Nahtbereich ({naht_mask.sum():,} Punkte, Z < -0.5mm):')
    print(f'  n_x  median={np.median(n_naht[:, 0]):+.3f}  mean={n_naht[:, 0].mean():+.3f}')
    print(f'  n_z  median={np.median(n_naht[:, 2]):+.3f}  mean={n_naht[:, 2].mean():+.3f}')
    print(f'  Anteil n_z > 0 (nach oben): {(n_naht[:, 2] > 0).mean()*100:.1f}%')
    print(f'  Anteil n_z < 0 (nach unten): {(n_naht[:, 2] < 0).mean()*100:.1f}%')

## 3 – Segmentation Pipeline (aus YAML)

Sanity-Check: lädt die komplette Pipeline aus `pipeline.yaml` und führt sie aus.


In [ ]:
seg_pipeline = SegmentationPipeline.from_config(CONFIG)
print(f'Pipeline: {seg_pipeline}')

labels_full, seg_report = seg_pipeline.process(pcd)
print()
print(seg_report.summary())


## 4 – Step-für-Step Visualisierung

Gleiche Steps manuell, um zwischen den Steps zu plotten. Parameter kommen aus der Config-geladenen Pipeline.


In [ ]:
# Punkte für XZ-Seitenansicht
pts = np.asarray(pcd.points)
n_points = len(pts)

def plot_xz(labels: np.ndarray, title: str, ax=None):
    """XZ-Seitenansicht mit Label-Färbung."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 4))
    colors = color_array(labels)
    # Reihenfolge: UNLABELED/Background unten, auffällige Labels oben
    order = np.argsort([{-1: 0, 0: 1, 4: 2, 3: 3, 1: 4, 2: 5}.get(int(l), 6) for l in labels])
    ax.scatter(pts[order, 0], pts[order, 2], c=colors[order], s=1, alpha=0.6)
    ax.set_xlabel('X (mm)')
    ax.set_ylabel('Z (mm)')
    ax.set_title(title)
    ax.set_aspect('equal')
    ax.grid(alpha=0.3)
    present = set(labels.tolist())
    ax.legend(handles=legend_handles(present), loc='lower right', fontsize=8,
              framealpha=0.9)
    return ax


### 4.1 – Initial (alles UNLABELED)

In [ ]:
labels = np.full(n_points, UNLABELED, dtype=np.int8)
fig, ax = plt.subplots(figsize=(12, 4))
plot_xz(labels, 'Initial – alles unlabeled', ax=ax)
plt.tight_layout(); plt.show()


### 4.2 – Nach BackgroundRemover

In [ ]:
step_bg = seg_pipeline.steps[0]  # BackgroundRemover
assert step_bg.name == 'background_remover'

labels, rep_bg = step_bg.apply(pcd, labels)

fig, ax = plt.subplots(figsize=(12, 4))
plot_xz(labels, f'Nach BackgroundRemover  ({rep_bg.duration_ms:.0f}ms)', ax=ax)
plt.tight_layout(); plt.show()

print(f'Tilt der Oberseiten-Ebene:  {rep_bg.artifacts["tilt_angle_deg"]:.2f}°')
print(f'Kandidaten:                 {rep_bg.artifacts["n_candidates"]:,}')
print(f'Inlier (→ background):      {rep_bg.artifacts["n_inliers"]:,}')
print(f'Verbleibend unlabeled:      {rep_bg.unlabeled_after:,}')


### 4.3 – Nach FlankSegmenter

In [ ]:
step_fl = seg_pipeline.steps[1]  # FlankSegmenter
assert step_fl.name == 'flank_segmenter'

labels, rep_fl = step_fl.apply(pcd, labels)

fig, ax = plt.subplots(figsize=(12, 4))
plot_xz(labels, f'Nach FlankSegmenter  ({rep_fl.duration_ms:.0f}ms)', ax=ax)
plt.tight_layout(); plt.show()

for side in ('flank_a', 'flank_b'):
    a = rep_fl.artifacts[side]
    print(f'{side}:  status={a["status"]}, '
          f'angle_from_vertical={a["angle_from_vertical_deg"]:.2f}°, '
          f'candidates={a["n_candidates"]:,}, inliers={a["n_inliers"]:,}')
    print(f'         plane_normal={np.round(a["plane_normal"], 3).tolist()}')


### 4.4 – Nach GapClassifier

In [ ]:
step_gap = seg_pipeline.steps[2]  # GapClassifier
assert step_gap.name == 'gap_classifier'

labels, rep_gap = step_gap.apply(pcd, labels)

# UNLABELED am Ende zu Background (wie in der Pipeline)
labels[labels == UNLABELED] = NAME_TO_ID['background']

fig, ax = plt.subplots(figsize=(12, 4))
plot_xz(labels, f'Nach GapClassifier + fill_background  ({rep_gap.duration_ms:.0f}ms)', ax=ax)
plt.tight_layout(); plt.show()

a = rep_gap.artifacts
print(f'x_bounds:   [{a["x_min"]:.2f}, {a["x_max"]:.2f}]')
print(f'z_lower:    {a["z_lower"]:.3f} mm  (unterhalb → sub_gap_artifacts)')
print(f'z_upper:    {a["z_upper"]:.3f} mm')
print(f'n_gap:      {a["n_gap"]:,}')
print(f'n_sub_gap:  {a["n_sub_gap"]:,}')


## 5 – Plausibilitätsmetriken

Soll-Werte der Werkstückgeometrie:
- Flankenwinkel zur Vertikalen: **30°** (nominelle 60° V-Naht)
- Spaltbreite entlang Y: **1.0mm → 2.5mm linear** (Dateiname: `Schweißspalt_1,0_auf_2,5`)
- Werkstück-Oberseite: **horizontal**


In [ ]:
# Labels-Verteilung
unique, counts = np.unique(labels, return_counts=True)
print(f'{"Label":<25}  {"Count":>10}  {"Anteil":>8}')
print('-' * 50)
for lid, cnt in zip(unique, counts):
    name = LABELS.get(int(lid), 'unlabeled')
    pct = 100 * cnt / n_points
    print(f'[{lid}] {name:<20}  {cnt:>10,}  {pct:>7.2f}%')


In [ ]:
# Spaltbreite entlang Y
widths = np.array(rep_gap.artifacts['gap_width_by_y'])
if widths.size:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(widths[:, 0], widths[:, 1], 'o-', color='#1976D2', label='Gemessen')
    # Soll-Linie: 1.0mm → 2.5mm linear entlang Y
    y_min, y_max = widths[:, 0].min(), widths[:, 0].max()
    ax.plot([y_min, y_max], [1.0, 2.5], '--', color='#E53935', label='Soll (1.0→2.5mm)')
    ax.set_xlabel('Y (mm)')
    ax.set_ylabel('Spaltbreite (mm)')
    ax.set_title('Spaltbreite entlang der Nahtrichtung')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

    print(f'Y-Bereich gemessen: [{widths[:, 0].min():.1f}, {widths[:, 0].max():.1f}] mm')
    print(f'Breite min/max:     [{widths[:, 1].min():.3f}, {widths[:, 1].max():.3f}] mm')
else:
    print('gap_width_by_y leer – zu wenige Flanken-Punkte pro Y-Slice.')


## 6 – Querschnitte bei verschiedenen Y-Positionen

Pro Y-Slice ein XZ-Querschnitt, farbig nach Label. Lässt erkennen, ob die Segmentation entlang der gesamten Nahtlänge konsistent bleibt und ob die Spaltbreite plausibel wächst.


In [ ]:
SLICE_THICKNESS = 0.5  # mm ±
N_SLICES = 6

y_min, y_max = pts[:, 1].min(), pts[:, 1].max()
y_positions = np.linspace(y_min + 5, y_max - 5, N_SLICES)

# Globale Achsenbereiche für konsistente Slice-Vergleiche
x_global = [pts[:, 0].min() - 1, pts[:, 0].max() + 1]
z_global = [pts[:, 2].min() - 0.5, pts[:, 2].max() + 0.5]

fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharex=True, sharey=True)
axes = axes.flatten()

for i, y in enumerate(y_positions):
    ax = axes[i]
    mask = np.abs(pts[:, 1] - y) < SLICE_THICKNESS
    sl_pts = pts[mask]
    sl_labels = labels[mask]
    colors = color_array(sl_labels)
    ax.scatter(sl_pts[:, 0], sl_pts[:, 2], c=colors, s=4, alpha=0.8)
    ax.set_title(f'Y ≈ {y:.0f} mm  ({mask.sum()} pts)')
    ax.set_xlim(x_global)
    ax.set_ylim(z_global)
    ax.set_aspect('equal')
    ax.grid(alpha=0.3)
    if i >= 3:
        ax.set_xlabel('X (mm)')
    if i % 3 == 0:
        ax.set_ylabel('Z (mm)')

# Gemeinsame Legende für alle Slices
fig.legend(handles=legend_handles(set(labels.tolist())),
           loc='upper center', ncol=5, bbox_to_anchor=(0.5, 1.02),
           fontsize=9, framealpha=0.9)
fig.suptitle(f'Querschnitte (±{SLICE_THICKNESS}mm)', y=1.05)
plt.tight_layout()
plt.show()


## 7 – Parameter-Tuning-Sandkasten

Drei Zellen zum Experimentieren mit einzelnen Steps. Jede Zelle startet frisch mit `labels = UNLABELED`, führt den jeweiligen Step (ggf. mit Vorgängern) mit alternativen Parametern aus und plottet das Ergebnis.

Wenn ein Parameter stabil gute Ergebnisse liefert: in `configs/pipeline.yaml` übertragen.


### 7.1 – BackgroundRemover tunen

In [ ]:
# Variiere ransac_threshold und normal_cos_threshold
step = BackgroundRemover(
    ransac_threshold=0.25,
    normal_cos_threshold=0.95,
)

lbl = np.full(n_points, UNLABELED, dtype=np.int8)
lbl, rep = step.apply(pcd, lbl)

fig, ax = plt.subplots(figsize=(12, 4))
plot_xz(lbl, f'BackgroundRemover  threshold={step.get_params()["ransac_threshold"]}, '
             f'cos_thr={step.get_params()["normal_cos_threshold"]}', ax=ax)
plt.tight_layout(); plt.show()

print(f'tilt={rep.artifacts["tilt_angle_deg"]:.2f}°  '
      f'inliers={rep.artifacts["n_inliers"]:,}  '
      f'remaining unlabeled={rep.unlabeled_after:,}')


### 7.2 – FlankSegmenter tunen (nach Background)

In [ ]:
# Variiere expected_flank_angle_deg und normal_cos_threshold
bg = BackgroundRemover(ransac_threshold=0.25, normal_cos_threshold=0.95)
fl = FlankSegmenter(
    ransac_threshold=0.25,
    expected_flank_angle_deg=45.0,   
    normal_cos_threshold=0.85,
)

lbl = np.full(n_points, UNLABELED, dtype=np.int8)
lbl, _ = bg.apply(pcd, lbl)
lbl, rep = fl.apply(pcd, lbl)

fig, ax = plt.subplots(figsize=(12, 4))
plot_xz(lbl, f'FlankSegmenter  expected={fl.get_params()["expected_flank_angle_deg"]}°, '
             f'cos_thr={fl.get_params()["normal_cos_threshold"]}', ax=ax)
plt.tight_layout(); plt.show()

for side in ('flank_a', 'flank_b'):
    a = rep.artifacts[side]
    print(f'{side}:  angle={a["angle_from_vertical_deg"]:.2f}°  '
          f'candidates={a["n_candidates"]:,}  inliers={a["n_inliers"]:,}  '
          f'status={a["status"]}')


### 7.3 – GapClassifier tunen (nach Background + Flanken)

In [ ]:
# Variiere z_lower_quantile, x_margin, separate_sub_gap_artifacts
bg  = BackgroundRemover(ransac_threshold=0.25, normal_cos_threshold=0.95)
fl  = FlankSegmenter(ransac_threshold=0.25, expected_flank_angle_deg=30.0,
                     normal_cos_threshold=0.85)
gap = GapClassifier(
    z_lower_quantile=0.05,
    x_margin=0.5,
    separate_sub_gap_artifacts=True,
)

lbl = np.full(n_points, UNLABELED, dtype=np.int8)
lbl, _ = bg.apply(pcd, lbl)
lbl, _ = fl.apply(pcd, lbl)
lbl, rep = gap.apply(pcd, lbl)
lbl[lbl == UNLABELED] = NAME_TO_ID['background']

fig, ax = plt.subplots(figsize=(12, 4))
plot_xz(lbl, f'GapClassifier  q={gap.get_params()["z_lower_quantile"]}, '
             f'margin={gap.get_params()["x_margin"]}', ax=ax)
plt.tight_layout(); plt.show()

a = rep.artifacts
print(f'z_lower={a["z_lower"]:.3f}  z_upper={a["z_upper"]:.3f}  '
      f'gap={a["n_gap"]:,}  sub_gap={a["n_sub_gap"]:,}')


## 8 – Visualisierung

Drei Varianten:
1. Plotly 3D – interaktiv, Klassen per Legend ein/aus
2. Querschnitts-Grid – 6 Y-Slices mit fixem Zoom
3. Y-Slider – einzelner Querschnitt mit verschiebbarem Y

### 8.1 – Interaktive 3D-Ansicht

Einfach-Klick auf Klasse in Legende blendet sie aus. Doppel-Klick isoliert. Ziehen dreht, Scroll zoomt.

In [ ]:
import plotly.graph_objects as go

def plot_3d_segmentation(labels, pcd=None, title='Segmentation', height=700,
                         point_size=1.8, opacity=0.75, max_points=None):
    """Interaktive 3D-Visualisierung der Segmentation."""
    if pcd is None:
        pcd = globals()['pcd']
    pts_v = np.asarray(pcd.points)

    if max_points is not None and len(pts_v) > max_points:
        idx = np.random.default_rng(0).choice(len(pts_v), max_points, replace=False)
        pts_v = pts_v[idx]
        labels = labels[idx]

    fig = go.Figure()
    # Feste Legend-Reihenfolge für Konsistenz
    label_order = [-1, 0, 4, 3, 1, 2]
    present = set(labels.tolist())
    for lid in label_order:
        if lid not in present:
            continue
        mask = labels == lid
        name = 'unlabeled' if lid == -1 else LABELS.get(int(lid), f'?{lid}')
        fig.add_trace(go.Scatter3d(
            x=pts_v[mask, 0], y=pts_v[mask, 1], z=pts_v[mask, 2],
            mode='markers',
            marker=dict(size=point_size, color=LABEL_COLORS[int(lid)], opacity=opacity),
            name=f'[{lid}] {name} ({mask.sum():,})',
        ))

    fig.update_layout(
        title=title,
        height=height,
        scene=dict(xaxis_title='X (mm)', yaxis_title='Y (mm)', zaxis_title='Z (mm)',
                   aspectmode='data'),
        legend=dict(itemsizing='constant', x=0.02, y=0.98),
        margin=dict(l=0, r=0, t=40, b=0),
    )
    fig.show()


plot_3d_segmentation(labels, title='Segmentation – komplette Pipeline')

### 8.2 – Querschnitts-Grid (XZ entlang Y)

6 Slices mit fixem X-Zoom auf die Naht-Region. Zeigt, ob die Segmentation entlang der Nahtlänge konsistent bleibt und ob die Spaltbreite plausibel zunimmt.

In [ ]:
def plot_cross_sections(labels, pcd=None, n_slices=6, slice_thickness=0.5,
                        point_size=8, figsize=(14, 7)):
    """XZ-Querschnitts-Grid, Zoom auf Naht-Region aus Flanken-Bounds."""
    if pcd is None:
        pcd = globals()['pcd']
    pts_cs = np.asarray(pcd.points)

    y_min, y_max = pts_cs[:, 1].min() + 5, pts_cs[:, 1].max() - 5
    y_positions = np.linspace(y_min, y_max, n_slices)

    # Zoom-Bereich aus Naht-Punkten ableiten
    naht_mask = np.isin(labels, [
        NAME_TO_ID['flank_a'], NAME_TO_ID['flank_b'],
        NAME_TO_ID['gap_region'], NAME_TO_ID['sub_gap_artifacts'],
    ])
    if naht_mask.any():
        x_center = np.median(pts_cs[naht_mask, 0])
        x_half = max(10.0, 1.5 * (pts_cs[naht_mask, 0].max() - pts_cs[naht_mask, 0].min()) / 2)
        x_lim = (x_center - x_half, x_center + x_half)
    else:
        x_lim = (pts_cs[:, 0].min(), pts_cs[:, 0].max())
    z_lim = (pts_cs[:, 2].min() - 0.5, max(1.0, pts_cs[:, 2].max() + 0.5))

    rows = (n_slices + 2) // 3
    fig, axes = plt.subplots(rows, 3, figsize=figsize, sharex=True, sharey=True,
                             squeeze=False)
    axes_flat = axes.flatten()

    for i, y in enumerate(y_positions):
        ax = axes_flat[i]
        mask = np.abs(pts_cs[:, 1] - y) < slice_thickness
        if not mask.any():
            ax.text(0.5, 0.5, 'Keine Punkte', transform=ax.transAxes, ha='center')
            ax.set_title(f'Y ≈ {y:.1f} mm')
            continue
        sl_pts = pts_cs[mask]
        sl_labels = labels[mask]
        colors = color_array(sl_labels)
        ax.scatter(sl_pts[:, 0], sl_pts[:, 2], c=colors, s=point_size, alpha=0.8)
        ax.set_title(f'Y ≈ {y:.1f} mm  ({mask.sum()} pts)')
        ax.set_xlim(x_lim); ax.set_ylim(z_lim)
        ax.set_aspect('equal'); ax.grid(alpha=0.3)

    for j in range(n_slices, len(axes_flat)):
        axes_flat[j].set_visible(False)

    for ax in axes[-1, :]:
        ax.set_xlabel('X (mm)')
    for ax in axes[:, 0]:
        ax.set_ylabel('Z (mm)')

    fig.legend(handles=legend_handles(set(labels.tolist())),
               loc='upper center', ncol=5, bbox_to_anchor=(0.5, 1.02),
               fontsize=9, framealpha=0.9)
    fig.suptitle(f'Querschnitte (±{slice_thickness}mm Slice-Dicke)', y=1.06)
    plt.tight_layout()
    plt.show()


plot_cross_sections(labels, n_slices=6)

### 8.3 – Interaktiver Y-Slider

Scrubbt durch die Nahtlänge. Erfordert `ipywidgets` (in Code-Server/VS Code normalerweise verfügbar).

In [ ]:
from ipywidgets import interact, FloatSlider

def _plot_single_slice(y, thickness):
    pts_s = np.asarray(pcd.points)
    mask = np.abs(pts_s[:, 1] - y) < thickness
    if not mask.any():
        print(f'Keine Punkte bei Y = {y:.2f} mm')
        return

    sl_pts = pts_s[mask]
    sl_labels = labels[mask]
    colors = color_array(sl_labels)

    naht_mask = np.isin(labels, [NAME_TO_ID['flank_a'], NAME_TO_ID['flank_b'],
                                 NAME_TO_ID['gap_region']])
    if naht_mask.any():
        x_center = np.median(pts_s[naht_mask, 0])
        x_lim = (x_center - 15, x_center + 15)
    else:
        x_lim = (pts_s[:, 0].min(), pts_s[:, 0].max())
    z_lim = (pts_s[:, 2].min() - 0.5, max(1.0, pts_s[:, 2].max() + 0.5))

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.scatter(sl_pts[:, 0], sl_pts[:, 2], c=colors, s=12, alpha=0.85)
    ax.set_xlim(x_lim); ax.set_ylim(z_lim)
    ax.set_aspect('equal'); ax.grid(alpha=0.3)
    ax.set_xlabel('X (mm)'); ax.set_ylabel('Z (mm)')
    ax.set_title(f'Y = {y:.2f} mm  (±{thickness:.2f}mm,  {mask.sum()} Punkte)')
    ax.legend(handles=legend_handles(set(sl_labels.tolist())),
              loc='lower right', fontsize=8)
    plt.tight_layout()
    plt.show()


_pts = np.asarray(pcd.points)
interact(
    _plot_single_slice,
    y=FloatSlider(min=float(_pts[:, 1].min()), max=float(_pts[:, 1].max()),
                  step=0.5, value=float(np.median(_pts[:, 1])),
                  description='Y (mm)', continuous_update=False, readout_format='.1f'),
    thickness=FloatSlider(min=0.1, max=2.0, step=0.1, value=0.5,
                          description='Dicke', continuous_update=False, readout_format='.1f'),
);

In [ ]:
# --- Diagnose: wie sieht's mit den Normalen pro Klasse im Rest-Scan aus? ---
normals = np.asarray(pcd.normals)
pts_d = np.asarray(pcd.points)

# Getrennt nach Z-Bereichen (Oberseite vs. Naht)
top_mask = pts_d[:, 2] > -0.3
naht_mask = pts_d[:, 2] < -0.3

print(f'Oberseite ({top_mask.sum():,} Punkte):')
print(f'  n_z > 0:                    {(normals[top_mask, 2] > 0).mean()*100:.1f}%')
print(f'  n_z > 0.95 (Background-Kandidat): {(normals[top_mask, 2] > 0.95).mean()*100:.1f}%')
print(f'  n_z median:                 {np.median(normals[top_mask, 2]):+.3f}')

print(f'\nNahtbereich ({naht_mask.sum():,} Punkte):')
print(f'  n_z > 0 (Flankenrichtung):  {(normals[naht_mask, 2] > 0).mean()*100:.1f}%')
print(f'  n_z < 0 (geflippt):         {(normals[naht_mask, 2] < 0).mean()*100:.1f}%')
print(f'  n_x median:                 {np.median(normals[naht_mask, 0]):+.3f}')
print(f'  n_z median:                 {np.median(normals[naht_mask, 2]):+.3f}')

# Kandidatenzählung bei verschiedenen Thresholds
print(f'\nFür FlankSegmenter (erwartet 45°, n_expected = (±0.707, 0, 0.707)):')
exp_a = np.array([0.707, 0, 0.707])
exp_b = np.array([-0.707, 0, 0.707])
cos_a = normals @ exp_a
cos_b = normals @ exp_b
for thr in [0.95, 0.9, 0.85, 0.75, 0.65]:
    print(f'  cos > {thr:.2f}:  A = {(cos_a > thr).sum():>5}  |  B = {(cos_b > thr).sum():>5}')

# Visualisierung im Naht-Bereich
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
# XZ-Ansicht, farbcodiert nach n_z
sc = axes[0].scatter(pts_d[naht_mask, 0], pts_d[naht_mask, 2],
                     c=normals[naht_mask, 2], cmap='RdBu', vmin=-1, vmax=1, s=3)
axes[0].set_title('Nahtbereich: n_z Farbe (blau=nach oben, rot=nach unten)')
axes[0].set_xlabel('X (mm)'); axes[0].set_ylabel('Z (mm)')
axes[0].set_aspect('equal')
plt.colorbar(sc, ax=axes[0])

# n_x im Naht-Bereich
sc = axes[1].scatter(pts_d[naht_mask, 0], pts_d[naht_mask, 2],
                     c=normals[naht_mask, 0], cmap='RdBu', vmin=-1, vmax=1, s=3)
axes[1].set_title('Nahtbereich: n_x Farbe (rot=rechts-zeigend, blau=links-zeigend)')
axes[1].set_xlabel('X (mm)'); axes[1].set_ylabel('Z (mm)')
axes[1].set_aspect('equal')
plt.colorbar(sc, ax=axes[1])
plt.tight_layout(); plt.show()

In [ ]:
# Falls noch nicht auf 45° korrigiert in pipeline.yaml, hier manuell:
fl = FlankSegmenter(
    ransac_threshold=0.25,
    expected_flank_angle_deg=45.0,
    normal_cos_threshold=0.85,
)

lbl = np.full(n_points, UNLABELED, dtype=np.int8)
lbl, _ = BackgroundRemover(ransac_threshold=0.25, normal_cos_threshold=0.95).apply(pcd, lbl)
lbl, rep = fl.apply(pcd, lbl)

for side in ('flank_a', 'flank_b'):
    a = rep.artifacts[side]
    status = a['status']
    print(f'{side}:  status={status}  candidates={a["n_candidates"]:,}  inliers={a["n_inliers"]:,}')
    if status == 'ok':
        print(f'         angle_from_vertical={a["angle_from_vertical_deg"]:.2f}°  '
              f'plane_normal={np.round(a["plane_normal"], 3).tolist()}')

In [ ]:
# Wie viele Kandidaten gibt es bei welchem Threshold?
normals_d = np.asarray(pcd.normals)
exp_a = np.array([np.cos(np.deg2rad(45)), 0, np.sin(np.deg2rad(45))])
exp_b = np.array([-np.cos(np.deg2rad(45)), 0, np.sin(np.deg2rad(45))])
cos_a = normals_d @ exp_a
cos_b = normals_d @ exp_b
for thr in [0.95, 0.9, 0.85, 0.8, 0.75, 0.7, 0.65, 0.6, 0.5]:
    print(f'cos > {thr:.2f}  (~±{int(np.degrees(np.arccos(thr)))}°):  '
          f'A = {(cos_a > thr).sum():>5}  |  B = {(cos_b > thr).sum():>5}')

In [ ]:
fl = FlankSegmenter(
    ransac_threshold=0.25,
    expected_flank_angle_deg=45.0,
    normal_cos_threshold=0.7,  # ~±45° Abweichung zugelassen
)

lbl = np.full(n_points, UNLABELED, dtype=np.int8)
lbl, _ = BackgroundRemover(ransac_threshold=0.25, normal_cos_threshold=0.95).apply(pcd, lbl)
lbl, rep = fl.apply(pcd, lbl)

for side in ('flank_a', 'flank_b'):
    a = rep.artifacts[side]
    print(f'{side}:  candidates={a["n_candidates"]:,}  inliers={a["n_inliers"]:,}  '
          f'angle={a.get("angle_from_vertical_deg", "?")}')

plot_3d_segmentation(lbl, title='Threshold 0.7')